# Sold Weeks 4-5

In [11]:
import pandas as pd

sold = pd.read_csv("../../IDX_Data/sold_with_rates.csv")
print("Initial shape:", sold.shape)
sold.head()

/var/folders/hx/s9px6scd2pl5dqfp0pldbdgc0000gn/T/ipykernel_71844/3013951622.py:3: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("../../IDX_Data/sold_with_rates.csv")


Initial shape: (542075, 86)


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,year_month,rate_30yr_fixed
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,NaN,NaN,90067,NaN,2105.00,177861.0,NaN,2220 Avenue Of The Stars 2704,NaN,NaN
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,3.0,Capistrano Unified,92677,NaN,254.00,5300.0,NaN,16 Palisades,NaN,NaN
2,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,2.0,NaN,91108,NaN,NaN,9404.0,NaN,1615 Waverly Road,NaN,NaN
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,...,4.0,Walnut Valley Unified,91765,NaN,295.95,58232.0,NaN,2250 Indian Creek Road,NaN,NaN
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaN,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,...,2.0,Newport Mesa Unified,92662,NaN,0.00,2250.0,NaN,317 E. Bayfront,NaN,NaN


In [12]:
date_cols = [
    'CloseDate',
    'PurchaseContractDate',
    'ListingContractDate',
    'ContractStatusChangeDate'
]

for col in date_cols:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors='coerce')

sold[date_cols].head()

,CloseDate,PurchaseContractDate,ListingContractDate,ContractStatusChangeDate
0,NaT,2024-05-07,2024-01-01,2024-05-07
1,NaT,NaT,2024-01-24,2024-01-24
2,NaT,NaT,2024-01-12,2024-01-12
3,NaT,NaT,2024-01-20,2024-01-20
4,NaT,NaT,2024-01-12,2024-01-12


In [13]:
missing_pct = sold.isnull().mean() * 100
high_missing_cols = missing_pct[missing_pct > 90].index.tolist()

print("Columns >90% missing:")
print(high_missing_cols)

Columns >90% missing:
['FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'BuilderName', 'TaxYear', 'BuildingAreaTotal', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict']


In [14]:
before_cols = sold.shape[1]
sold = sold.drop(columns=high_missing_cols)
after_cols = sold.shape[1]

print("Columns before:", before_cols)
print("Columns after:", after_cols)

Columns before: 86
Columns after: 73


In [15]:
numeric_cols = [
    'ClosePrice',
    'LivingArea',
    'DaysOnMarket',
    'BedroomsTotal',
    'BathroomsTotalInteger'
]

for col in numeric_cols:
    if col in sold.columns:
        sold[col] = pd.to_numeric(sold[col], errors='coerce')

sold[numeric_cols].dtypes

ClosePrice               float64
LivingArea               float64
DaysOnMarket               int64
BedroomsTotal            float64
BathroomsTotalInteger    float64
dtype: object

In [16]:
before_rows = sold.shape[0]
sold = sold[
    (sold['ClosePrice'] > 0) &
    (sold['LivingArea'] > 0) &
    (sold['DaysOnMarket'] >= 0) &
    (sold['BedroomsTotal'] >= 0) &
    (sold['BathroomsTotalInteger'] >= 0)
]
after_rows = sold.shape[0]

print("Rows before:", before_rows)
print("Rows after:", after_rows)

Rows before: 542075
Rows after: 140752


In [17]:
sold['listing_after_close_flag'] = (
    sold['ListingContractDate'] > sold['CloseDate']
)

sold['purchase_after_close_flag'] = (
    sold['PurchaseContractDate'] > sold['CloseDate']
)

sold['negative_timeline_flag'] = (
    (sold['ListingContractDate'] > sold['PurchaseContractDate']) |
    (sold['PurchaseContractDate'] > sold['CloseDate'])
)

In [18]:
print("Listing after close:", sold['listing_after_close_flag'].sum())
print("Purchase after close:", sold['purchase_after_close_flag'].sum())
print("Negative timeline:", sold['negative_timeline_flag'].sum())

Listing after close: 71
Purchase after close: 245
Negative timeline: 459


In [19]:
missing_coords = sold['Latitude'].isnull() | sold['Longitude'].isnull()
zero_coords = (sold['Latitude'] == 0) | (sold['Longitude'] == 0)
positive_longitude = sold['Longitude'] > 0

print("Missing coords:", missing_coords.sum())
print("Zero coords:", zero_coords.sum())
print("Positive longitude:", positive_longitude.sum())

Missing coords: 15781
Zero coords: 6
Positive longitude: 9


In [20]:
sold.to_csv("sold_cleaned_week4_5.csv", index=False)
print("Cleaned SOLD dataset saved.")

Cleaned SOLD dataset saved.


### Data Cleaning Summary

- Converted date fields to datetime format
- Removed columns with more than 90% missing values
- Ensured numeric fields were properly typed
- Removed invalid numeric values (e.g., negative or zero values)
- Created date consistency flags
- Performed geographic data quality checks

### Validation Summary

- Row counts before and after filtering were tracked
- Date inconsistencies were identified using flag variables
- Geographic anomalies were flagged (missing or invalid coordinates)
- Final dataset is cleaned and ready for analysis